In [1]:
import torch
from torch import nn
import torch.optim as optim
import torch.nn as nn
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from dateutil.relativedelta import relativedelta
import geopandas as gpd
from shapely.geometry import Polygon, Point
from sklearn.metrics import mean_squared_error

In [2]:
df = pd.read_csv("../../../Data/Final/Imputation/changi_imp_final.csv")
print(df.head())

            x         y          Date      Value
0  103.964566  1.350459  Mar-Apr 2000  23.775064
1  103.964566  1.351269  Mar-Apr 2000  23.403744
2  103.964566  1.352079  Mar-Apr 2000  22.966872
3  103.964566  1.352889  Mar-Apr 2000  22.417757
4  103.965377  1.348838  Mar-Apr 2000  23.699749


In [3]:
## THIS CODE IS TO REFORMAT THE TIME INDEX IN THE DATASET ##

# Function to create a time index
def create_time_index(df):
    bimonthly_map = {"Jan-Feb": 1, "Mar-Apr": 2, "May-Jun": 3, "Jul-Aug": 4, "Sep-Oct": 5, "Nov-Dec": 6}
    
    df["Year"] = df["Date"].str[-4:].astype(int)  # Extract year
    df["Period"] = df["Date"].str[:-5].map(bimonthly_map)  # Extract period and map it

    df["time_index"] = (df["Year"] - 2000) * 6 + df["Period"]  # Compute time index
    df = df.drop(columns=["Year", "Period"])  # Drop extra columns
    
    return df

# Apply time index transformation
df = create_time_index(df)

In [4]:
## THIS CODE IS TO FILTER A RANDOM SAMPLE OF 100 COORDINATES INSIDE A BOUNDING BOX ##

np.random.seed(5188)  # Set random seed for reproducibility

# Define the bounding box (replace with actual values)
coordinates = [103.98, 104.05, 1.32, 1.38]  # Example bounding box for Changi
bbox = Polygon([
    (coordinates[0], coordinates[2]), 
    (coordinates[1], coordinates[2]), 
    (coordinates[1], coordinates[3]), 
    (coordinates[0], coordinates[3]), 
    (coordinates[0], coordinates[2])
])

# Function to filter 100 coordinates inside bounding box
def filter_by_geography(df, bbox, seed=5188, n=100):
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["x"], df["y"]), crs="EPSG:4326")
    filtered_gdf = gdf[gdf.within(bbox)]
    
    # Select 100 unique coordinates
    sampled_coords = filtered_gdf[['x', 'y']].drop_duplicates().sample(n=n, random_state=seed)
    filtered_df = df.merge(sampled_coords, on=['x', 'y'])
    
    return filtered_df

# Apply filtering
df = filter_by_geography(df, bbox)

# Keep only relevant columns
df = df[["x", "y", "Date", "Value", "time_index"]]

# # Save the filtered dataset
# df.to_csv("filtered_changi.csv", index=False)
# print("Filtered dataset saved as filtered_changi.csv")

## To ensure that the same 100 coordinates are used check the unique (x,y) pairs in the filtered dataset ##
unique_coords = df[['x', 'y']].drop_duplicates()
print(f"Number of unique coordinates: {len(unique_coords)}")

Number of unique coordinates: 100


In [5]:
## THIS CODE IS TO EXTRACT ALL POSSIBLE WINDOWS OF SIZE = 13 ##

# Define sequence length
sequence_length = 13 * 6
target_length = 12  # Predict the next 12 time steps

# Group by (x, y) and process each time series separately
grouped = df.groupby(["x", "y"])

# Lists to store input sequences and target sequences separately
input_sequences = []
target_sequences = []
locations = []

# Iterate over each coordinate group
for (x, y), group in grouped:
    # Sort by time index
    group = group.sort_values(by="time_index")

    # Extract LST values
    values = group["Value"].values

    # Generate sequences
    for i in range(len(values) - sequence_length - target_length + 1):
        input_seq = values[i : i + sequence_length]  # Past values
        target_seq = values[i + sequence_length : i + sequence_length + target_length]  # Next 12 values
        
        # Store sequences separately
        input_sequences.append([x, y] + list(input_seq))
        target_sequences.append([x, y] + list(target_seq))
        locations.append([x, y])

# Define column names
input_columns = ["x", "y"] + [f"LST_t-{i}" for i in range(sequence_length, 0, -1)]
target_columns = ["x", "y"] + [f"Target_t+{i}" for i in range(1, target_length + 1)]

# Convert to DataFrames
input_df = pd.DataFrame(input_sequences, columns=input_columns)
target_df = pd.DataFrame(target_sequences, columns=target_columns)

# # Save input and target sequences as separate CSV files
# input_df.to_csv("changi_inputs_9.csv", index=False)
# target_df.to_csv("changi_targets_10.csv", index=False)


In [7]:
## THIS FUNCTION IS TO CREATE THE 10 RANDOM TRAIN-TEST SAMPLES ##

np.random.seed(5188)

# Function to create a time index
def create_time_index(df):
    bimonthly_map = {"Jan-Feb": 1, "Mar-Apr": 2, "May-Jun": 3, "Jul-Aug": 4, "Sep-Oct": 5, "Nov-Dec": 6}
    df["Year"] = df["Date"].str[-4:].astype(int)  # Extract year
    df["Period"] = df["Date"].str[:-5].map(bimonthly_map)  # Extract period and map it
    df["time_index"] = (df["Year"] - 2000) * 6 + df["Period"]  # Compute time index
    df = df.drop(columns=["Year", "Period"])  # Drop extra columns
    return df

df = create_time_index(df)

# Select 100 random unique coordinate pairs
unique_coords = df[['x', 'y']].drop_duplicates().sample(n=100, random_state=5188)
selected_df = df.merge(unique_coords, on=['x', 'y'])

# Get valid start indices for 13-year training windows
min_time = selected_df["time_index"].min()
max_time = selected_df["time_index"].max()
valid_start_times = np.arange(min_time, max_time - (13 * 6 + 2 * 6) + 1)

# Randomly sample 10 different start times
sampled_start_times = np.random.choice(valid_start_times, 10, replace=False)

# Extract train and test sets for each sampled start time
samples = []
for start_time in sampled_start_times:
    train_end_time = start_time + (13 * 6) - 1
    test_end_time = train_end_time + (2 * 6)
    
    train_set = selected_df[(selected_df["time_index"] >= start_time) & (selected_df["time_index"] <= train_end_time)]
    test_set = selected_df[(selected_df["time_index"] > train_end_time) & (selected_df["time_index"] <= test_end_time)]
    
    samples.append((train_set, test_set))


# # Save sampled train-test sets
# for i, (train_set, test_set) in enumerate(samples):
#     train_set.to_csv(f"train_sample_{i+1}.csv", index=False)
#     test_set.to_csv(f"test_sample_{i+1}.csv", index=False)


### Without dropout

In [5]:
## THIS CODE IS TO CREATE THE LSTM ##

class LSTMPredictor(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=3, output_size=1):
        super(LSTMPredictor, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        out = self.fc(lstm_out)
        return out

In [ ]:
## THIS CODE IS TO TRAIN THE LSTM ##
torch.manual_seed(5188)

# Function to prepare sequences for LSTM
def create_lstm_sequences(series, seq_length):
    X, y = [], []
    for i in range(len(series) - seq_length):
        X.append(series[i:i+seq_length])
        y.append(series[i+1:i+seq_length+1])
    return np.array(X), np.array(y)

# Training settings
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_epochs = 100
seq_length = 12

# Initialize RMSE tracker
rmse_stats = {
    "1st": [], "3rd": [], "6th": [], "9th": [], "12th": [], "Overall": []
}

# Iterate over every sampled train-test set
for sample_idx, (train_set, test_set) in enumerate(samples):
    
    value_series = train_set['Value'].values.astype(np.float32)
    test_values = test_set['Value'].values.astype(np.float32)[:12]  # Only next 12 values

    mean, std = value_series.mean(), value_series.std()
    value_series_norm = (value_series - mean) / std

    X_seq, y_seq = create_lstm_sequences(value_series_norm.reshape(-1, 1), seq_length)
    X_train = torch.tensor(X_seq, dtype=torch.float32).to(device)
    y_train = torch.tensor(y_seq, dtype=torch.float32).to(device)

    model = LSTMPredictor().to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()

    for epoch in range(num_epochs):
        model.train()
        optimizer.zero_grad()
        output = model(X_train)
        loss = criterion(output[:, :-1, :], y_train[:, 1:, :])
        loss.backward()
        optimizer.step()

        if epoch % 10 == 0:
            print(f"Sample {sample_idx+1}, Epoch {epoch}, Loss: {loss.item():.4f}")

    model.eval()
    with torch.no_grad():
        input_seq = torch.tensor(value_series_norm[-seq_length:].reshape(1, seq_length, 1), dtype=torch.float32).to(device)
        preds = []
        for _ in range(12):
            pred = model(input_seq)[:, -1:, :]
            preds.append(pred.item())
            input_seq = torch.cat((input_seq[:, 1:, :], pred), dim=1)

    preds = np.array(preds) * std + mean

    # Calculate RMSEs
    for step, label in zip([0, 2, 5, 8, 11], ["1st", "3rd", "6th", "9th", "12th"]):
        rmse = np.sqrt(mean_squared_error([test_values[step]], [preds[step]]))
        rmse_stats[label].append(rmse)

    all_rmse = np.sqrt(mean_squared_error(test_values, preds))
    rmse_stats["Overall"].append(all_rmse)

# Compute average RMSE across samples
rmse_df_13 = pd.DataFrame({k: [np.mean(v)] for k, v in rmse_stats.items()}).T
rmse_df_13.columns = ["Window = 13"]
rmse_df_13.index.name = "Forecast horizon"
print("\nAverage RMSE per forecast step:")
print(rmse_df_13)
rmse_df_13.to_csv("rmse_13.csv")

Sample 1, Epoch 0, Loss: 1.0038
Sample 1, Epoch 10, Loss: 0.9077
Sample 1, Epoch 20, Loss: 0.5168
Sample 1, Epoch 30, Loss: 0.4792
Sample 1, Epoch 40, Loss: 0.4476
Sample 1, Epoch 50, Loss: 0.4013
Sample 1, Epoch 60, Loss: 0.3547
Sample 1, Epoch 70, Loss: 0.3117
Sample 1, Epoch 80, Loss: 0.2884
Sample 1, Epoch 90, Loss: 0.2735
Sample 2, Epoch 0, Loss: 1.0009
Sample 2, Epoch 10, Loss: 0.9157
Sample 2, Epoch 20, Loss: 0.5372
Sample 2, Epoch 30, Loss: 0.4987
Sample 2, Epoch 40, Loss: 0.4658
Sample 2, Epoch 50, Loss: 0.4165
Sample 2, Epoch 60, Loss: 0.3664
Sample 2, Epoch 70, Loss: 0.3207
Sample 2, Epoch 80, Loss: 0.2953
Sample 2, Epoch 90, Loss: 0.2784
Sample 3, Epoch 0, Loss: 1.0076
Sample 3, Epoch 10, Loss: 0.9389
Sample 3, Epoch 20, Loss: 0.6064
Sample 3, Epoch 30, Loss: 0.5303
Sample 3, Epoch 40, Loss: 0.4646
Sample 3, Epoch 50, Loss: 0.4030
Sample 3, Epoch 60, Loss: 0.3451
Sample 3, Epoch 70, Loss: 0.3050
Sample 3, Epoch 80, Loss: 0.2827
Sample 3, Epoch 90, Loss: 0.2667
Sample 4, Epo

### With dropout

In [8]:
## THIS CODE IS TO CREATE THE LSTM ##

class LSTMPredictor(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=3, output_size=1, dropout=0.2):
        super(LSTMPredictor, self).__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers, 
            batch_first=True, dropout=dropout
        )
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        out = self.fc(lstm_out)
        return out


In [9]:
## THIS CODE IS TO TRAIN THE LSTM ##
torch.manual_seed(5188)

# Function to prepare sequences for LSTM
def create_lstm_sequences(series, seq_length):
    X, y = [], []
    for i in range(len(series) - seq_length):
        X.append(series[i:i+seq_length])
        y.append(series[i+1:i+seq_length+1])
    return np.array(X), np.array(y)

# Training settings
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_epochs = 100
seq_length = 12

# Initialize RMSE tracker
rmse_stats = {
    "1st": [], "3rd": [], "6th": [], "9th": [], "12th": [], "Overall": []
}

# Iterate over each sampled train-test set
for sample_idx, (train_set, test_set) in enumerate(samples):
    
    value_series = train_set['Value'].values.astype(np.float32)
    test_values = test_set['Value'].values.astype(np.float32)[:12]  # Only next 12 values

    mean, std = value_series.mean(), value_series.std()
    value_series_norm = (value_series - mean) / std

    X_seq, y_seq = create_lstm_sequences(value_series_norm.reshape(-1, 1), seq_length)
    X_train = torch.tensor(X_seq, dtype=torch.float32).to(device)
    y_train = torch.tensor(y_seq, dtype=torch.float32).to(device)

    model = LSTMPredictor().to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-6)
    criterion = nn.MSELoss()

    for epoch in range(num_epochs):
        model.train()
        optimizer.zero_grad()
        output = model(X_train)
        loss = criterion(output[:, :-1, :], y_train[:, 1:, :])
        loss.backward()
        optimizer.step()

        if epoch % 10 == 0:
            print(f"Sample {sample_idx+1}, Epoch {epoch}, Loss: {loss.item():.4f}")

    model.eval()
    with torch.no_grad():
        input_seq = torch.tensor(value_series_norm[-seq_length:].reshape(1, seq_length, 1), dtype=torch.float32).to(device)
        preds = []
        for _ in range(12):
            pred = model(input_seq)[:, -1:, :]
            preds.append(pred.item())
            input_seq = torch.cat((input_seq[:, 1:, :], pred), dim=1)

    preds = np.array(preds) * std + mean

    # Calculate RMSEs
    for step, label in zip([0, 2, 5, 8, 11], ["1st", "3rd", "6th", "9th", "12th"]):
        rmse = np.sqrt(mean_squared_error([test_values[step]], [preds[step]]))
        rmse_stats[label].append(rmse)

    all_rmse = np.sqrt(mean_squared_error(test_values, preds))
    rmse_stats["Overall"].append(all_rmse)

# Compute average RMSE across samples
rmse_df_13_dropout = pd.DataFrame({k: [np.mean(v)] for k, v in rmse_stats.items()}).T
rmse_df_13_dropout.columns = ["Window = 13"]
rmse_df_13_dropout.index.name = "Forecast horizon"
print("\nAverage RMSE per forecast step:")
print(rmse_df_13_dropout)
rmse_df_13_dropout.to_csv("rmse_dropout_13.csv")

Sample 1, Epoch 0, Loss: 1.0038
Sample 1, Epoch 10, Loss: 0.9080
Sample 1, Epoch 20, Loss: 0.5179
Sample 1, Epoch 30, Loss: 0.4807
Sample 1, Epoch 40, Loss: 0.4492
Sample 1, Epoch 50, Loss: 0.4027
Sample 1, Epoch 60, Loss: 0.3571
Sample 1, Epoch 70, Loss: 0.3158
Sample 1, Epoch 80, Loss: 0.2925
Sample 1, Epoch 90, Loss: 0.2789
Sample 2, Epoch 0, Loss: 1.0010
Sample 2, Epoch 10, Loss: 0.9162
Sample 2, Epoch 20, Loss: 0.5388
Sample 2, Epoch 30, Loss: 0.5002
Sample 2, Epoch 40, Loss: 0.4679
Sample 2, Epoch 50, Loss: 0.4182
Sample 2, Epoch 60, Loss: 0.3693
Sample 2, Epoch 70, Loss: 0.3238
Sample 2, Epoch 80, Loss: 0.2998
Sample 2, Epoch 90, Loss: 0.2841
Sample 3, Epoch 0, Loss: 1.0076
Sample 3, Epoch 10, Loss: 0.9391
Sample 3, Epoch 20, Loss: 0.6078
Sample 3, Epoch 30, Loss: 0.5320
Sample 3, Epoch 40, Loss: 0.4669
Sample 3, Epoch 50, Loss: 0.4058
Sample 3, Epoch 60, Loss: 0.3487
Sample 3, Epoch 70, Loss: 0.3098
Sample 3, Epoch 80, Loss: 0.2880
Sample 3, Epoch 90, Loss: 0.2728
Sample 4, Epo